# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malehamajid/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [22]:
%pip install -q duckdb huggingface_hub pandas scikit-learn

In [23]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

BASE = "hf://datasets/FlyRank/internship-warehouse"

DEV_MONTH = "2026-03"
SEALED_MONTH = "2026-06"

In [24]:
con.sql(f"""
SELECT COUNT(*) AS rows
FROM read_parquet(
'{BASE}/fact_content_daily_performance/month={DEV_MONTH}/*.parquet'
)
""").show()

┌─────────┐
│  rows   │
│  int64  │
├─────────┤
│ 9841378 │
└─────────┘



In [25]:
con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet(
'{BASE}/fact_content_daily_performance/month={DEV_MONTH}/*.parquet'
)
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



- One row represents one content item for one client on one reporting date.

- The unit of analysis (grain) is:
  report_date × client_hash_id × content_hash_id

- The table used for this analysis is:
  fact_content_daily_performance

- The development time window is:
  month=2026-03

- March 2026 is selected as a mid-panel month so that the model development does not use the final sealed test month (2026-06).

- The goal of this lane is to identify content pages that are potential refresh opportunities based on their historical search performance.

- The prediction target is a refresh opportunity proxy based on future performance improvement.

- Data from the future period is excluded from features because it would create data leakage.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*



### Features (known at decision moment)

- gsc_clicks: Historical search clicks observed before the decision date.
- gsc_impressions: Historical search impressions observed before the decision date.
- ctr: Calculated from historical clicks and impressions only.
- gsc_sum_position: Historical search ranking signal available from Search Console data.
- sessions_ai: Historical AI traffic/session information available before prediction time.

### Label / Proxy

- Refresh opportunity score:
  A proxy label representing whether a content item shows potential for improvement based on future performance changes.

- The label is created using future performance data only and is not used as a feature.

### Context

- report_date: Date of the performance record.
- client_hash_id: Anonymous identifier for the client.
- content_hash_id: Anonymous identifier for the content item.
- month: Dataset partition month.

### Excluded

- Future performance fields: excluded because they would leak information from the prediction period.
- Client and content identifiers are used only for grouping and analysis, not as predictive features.
- Any fields that are not available at the decision moment are excluded to keep the evaluation realistic.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain verification

Claim:
One row represents one content item for one client on one reporting date.

The query checks whether the combination of report_date, client_hash_id, and content_hash_id is unique.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT(
        report_date,
        client_hash_id,
        content_hash_id
    )) AS unique_grain_rows
FROM read_parquet(
    '{BASE}/fact_content_daily_performance/month={DEV_MONTH}/*.parquet'
)
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────────┐
│ total_rows │ unique_grain_rows │
│   int64    │       int64       │
├────────────┼───────────────────┤
│    9841378 │           9841378 │
└────────────┴───────────────────┘



### Query 2 — Row count and date range

Claim:
This query confirms the size of the March 2026 slice and the available reporting window.

In [29]:
con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM read_parquet(
    '{BASE}/fact_content_daily_performance/month={DEV_MONTH}/*.parquet'
)
""").show()

┌───────────┬────────────┬────────────┬───────────────┐
│ row_count │ start_date │  end_date  │ content_items │
│   int64   │    date    │    date    │     int64     │
├───────────┼────────────┼────────────┼───────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │        331437 │
└───────────┴────────────┴────────────┴───────────────┘



### Query 3 — GSC availability check

Claim:
Only records where GSC data is available should be used for search performance features.

In [30]:
con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN 1
            ELSE 0
        END
    ) AS available_rows
FROM read_parquet(
    '{BASE}/fact_content_daily_performance/month={DEV_MONTH}/*.parquet'
)
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────┐
│ total_rows │ available_rows │
│   int64    │     int128     │
├────────────┼────────────────┤
│    9841378 │        3611061 │
└────────────┴────────────────┘



### Feature frame

The feature frame contains only information available at the decision moment.

In [31]:
feature_frame = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,

    gsc_clicks,
    gsc_impressions,

    CASE
        WHEN gsc_impressions > 0
        THEN gsc_clicks * 1.0 / gsc_impressions
        ELSE 0
    END AS ctr,

    gsc_sum_position,
    sessions_ai

FROM read_parquet(
'{BASE}/fact_content_daily_performance/month={DEV_MONTH}/*.parquet'
)

WHERE gsc_data_available IS TRUE
LIMIT 1000
""").df()

feature_frame.head()

,report_date,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,ctr,gsc_sum_position,sessions_ai
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,0,20,0.000,67,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,0,1,0.000,0,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,1,125,0.008,616,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,0,7,0.000,28,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,0,11,0.000,25,<NA>


## The trap — deliberate feature leakage

A future performance field is intentionally added as a feature to demonstrate data leakage.

This feature contains information that would not be available at the decision moment, but it is directly related to the outcome.

The model performance improves unrealistically when the leaked feature is included.

After the experiment, the leaked feature is removed and only decision-time available features are kept.

In [32]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

df = feature_frame.copy()

# Missing values handle karo
df = df.fillna(0)

# Leakage feature intentionally add karo
df["future_clicks"] = df["gsc_clicks"].shift(-1)

# Last missing future value remove
df = df.dropna(subset=["future_clicks"])

# Create proxy label
df["refresh_opportunity"] = (
    df["future_clicks"] > df["gsc_clicks"]
).astype(int)


honest_features = [
    "gsc_clicks",
    "gsc_impressions",
    "ctr",
    "gsc_sum_position",
    "sessions_ai"
]


# ---------------- WITH LEAK ----------------

X = df[honest_features + ["future_clicks"]]
y = df["refresh_opportunity"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

pred = model.predict_proba(X_test)[:,1]

print(
    "AUC with leakage:",
    roc_auc_score(y_test, pred)
)


# ---------------- WITHOUT LEAK ----------------

X = df[honest_features]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict_proba(X_test)[:,1]

print(
    "AUC without leakage:",
    roc_auc_score(y_test, pred)
)

AUC with leakage: 0.9963086009597638
AUC without leakage: 0.688030638612034


## Leakage lesson

The leaked feature increased model performance because it contained future information from the outcome period.

This information would not be available when making a real refresh decision.

The leaked feature was removed, and the final feature set only contains information available at the decision moment.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

One limitation of this data is unbalanced history across clients.

Different clients may have different GSC availability periods, which means some content items may have less historical data available.

This can affect features that depend on past performance windows because newer or incomplete records may not have the same amount of historical information.

Therefore, results should be interpreted as directional decision-support insights rather than guaranteed causal conclusions.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.